# 01 Data and study design

The experiment evaluates voltage and active-loss prediction, voltage feasibility, and candidate-action ranking in radial distribution networks. Each observation is an independently simulated balanced steady-state load, topology and source-voltage configuration.

Targets are minimum bus voltage (p.u.), active losses (kW), and feasibility under the 0.95–1.05 p.u. voltage criterion. Equipment thermal ratings are unavailable. The working hypothesis is that graph connectivity and a physical reference improve prediction across electrical configurations and telemetry conditions.

Training, model selection, probability calibration and testing use disjoint topologies. The test set was examined during model development; its results are exploratory rather than a blinded confirmation. Hyperparameter selection uses the selection set. The external feeder evaluates transfer to a different network size.

The input comprises benchmark electrical parameters and simulated telemetry. Subsequent notebooks use the resulting features, targets and training-fold assignments. The data contain no field SCADA measurements or chronological operating trajectories.

Benchmark sources: [MATPOWER case33bw](https://matpower.org/docs/ref/matpower6.0/case33bw.html) and the case69 source recorded with the local benchmark parameters.

In [1]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
assert (PROJECT / "src" / "models.py").is_file()
sys.path.insert(0, str(PROJECT))
from src.notebook_tools import configure_style, show_table, save_figure

from src.protocol import SEED, SEEDS, FULL_CONFIG, TRAINING, MODEL_ORDER

configure_style()
rng = np.random.default_rng(SEED)
FIGURES = PROJECT / "figures"
TABLES = PROJECT / "artifacts" / "generated" / "tables"
RESULTS = PROJECT / "artifacts" / "generated" / "results"
MODELS = PROJECT / "artifacts" / "generated" / "models"
for directory in (FIGURES, TABLES, RESULTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

In [2]:
from src.benchmarks import case33, case69, RadialBenchmark
from src.gridstudy import (
    Topology,
    enumerate_case33_topologies,
    topology_edges,
    orient_tree,
    generate_load_state,
    corrupt_telemetry,
    scenario_tabular_features,
    build_node_features,
    graph_matrices,
)
from src.models import load_bundle
from sklearn.model_selection import GroupKFold

## Physical model and scenario generation

Backward/forward sweep solves balanced radial constant-power states. Nonconverged states have no accepted voltage or loss targets; they remain labeled inadmissible for classification. Nonconvergence alone does not prove physical infeasibility.

In [3]:
def radial_power_flow(
    benchmark: RadialBenchmark,
    p_kw: np.ndarray,
    q_kvar: np.ndarray,
    topology: Topology | None = None,
    slack_vm: float = 1.0,
    max_iter: int = 100,
    tol: float = 1e-10,
) -> dict[str, np.ndarray | float | bool | int]:
    n_bus = len(benchmark.bus_p_kw)
    edges, r_ohm, x_ohm = topology_edges(benchmark, topology)
    parent, parent_edge, order = orient_tree(n_bus, edges)
    z_base = (benchmark.base_kv * 1e3) ** 2 / (benchmark.base_mva * 1e6)
    z_pu = (r_ohm + 1j * x_ohm) / z_base
    s_pu = (np.asarray(p_kw) + 1j * np.asarray(q_kvar)) / (benchmark.base_mva * 1000.0)
    v = np.ones(n_bus, dtype=complex) * complex(slack_vm, 0.0)
    branch_current = np.zeros(len(edges), dtype=complex)

    converged = False
    for iteration in range(1, max_iter + 1):
        safe_v = np.where(np.abs(v) < 1e-8, 1 + 0j, v)
        i_bus = np.conj(s_pu / safe_v)
        downstream = i_bus.copy()
        branch_current.fill(0.0)
        for node in order[:0:-1]:
            edge_index = parent_edge[node]
            branch_current[edge_index] = downstream[node]
            downstream[parent[node]] += downstream[node]
        v_new = np.empty_like(v)
        v_new[0] = complex(slack_vm, 0.0)
        for node in order[1:]:
            edge_index = parent_edge[node]
            v_new[node] = v_new[parent[node]] - z_pu[edge_index] * branch_current[edge_index]
        if np.max(np.abs(v_new - v)) < tol:
            v = v_new
            converged = True
            break
        v = v_new

    losses_kw = float(
        np.sum((r_ohm / z_base) * np.abs(branch_current) ** 2) * benchmark.base_mva * 1000.0
    )
    vm = np.abs(v)
    return {
        "vm": vm,
        "vmin": float(vm.min()),
        "vmax": float(vm.max()),
        "loss_kw": losses_kw,
        "converged": converged,
        "iterations": iteration,
        "branch_current_pu": np.abs(branch_current),
    }

In [4]:
def make_dataset(
    output_dir: str | Path,
    n_case33: int = 5000,
    n_case69: int = 1200,
    seed: int = 20260922,
    max_nodes: int = 69,
) -> dict[str, int]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    benchmark33 = case33()
    benchmark69 = case69()
    topologies = enumerate_case33_topologies()
    topology_ids = np.arange(len(topologies))
    rng.shuffle(topology_ids)
    n_train_top = max(1, int(0.55 * len(topologies)))
    n_selection_top = max(1, int(0.15 * len(topologies)))
    n_calibration_top = max(1, int(0.15 * len(topologies)))
    train_top = set(topology_ids[:n_train_top].tolist())
    selection_top = set(topology_ids[n_train_top : n_train_top + n_selection_top].tolist())
    calibration_top = set(
        topology_ids[
            n_train_top + n_selection_top : n_train_top + n_selection_top + n_calibration_top
        ].tolist()
    )
    test_top = set(topology_ids[n_train_top + n_selection_top + n_calibration_top :].tolist())

    feature_names: list[str] | None = None
    records: list[dict[str, float | int | str]] = []
    node_arrays: list[np.ndarray] = []
    adj_arrays: list[np.ndarray] = []
    node_masks: list[np.ndarray] = []
    tab_arrays: list[np.ndarray] = []
    targets: list[np.ndarray] = []

    for sample_id in range(n_case33):
        topology = topologies[int(rng.integers(0, len(topologies)))]
        p_true, q_true, state = generate_load_state(benchmark33, rng)
        slack_vm = float(rng.uniform(0.975, 1.045))
        exact = radial_power_flow(benchmark33, p_true, q_true, topology, slack_vm)
        p_obs, q_obs, obs_mask, age = corrupt_telemetry(p_true, q_true, rng)
        tab, tab_dict = scenario_tabular_features(
            benchmark33, topology, p_obs, q_obs, obs_mask, age, slack_vm
        )
        if feature_names is None:
            feature_names = list(tab_dict.keys())
        node = build_node_features(
            benchmark33, topology, p_obs, q_obs, obs_mask, age, max_nodes
        )
        adj, _, nmask = graph_matrices(benchmark33, topology, max_nodes)
        if topology.topology_id in train_top:
            split = "train"
        elif topology.topology_id in selection_top:
            split = "selection"
        elif topology.topology_id in calibration_top:
            split = "calibration"
        else:
            split = "test"
        feasible = int(
            bool(exact["converged"]) and exact["vmin"] >= 0.95 and exact["vmax"] <= 1.05
        )
        records.append(
            {
                "sample_id": sample_id,
                "network": "case33bw",
                "split": split,
                "topology_id": topology.topology_id,
                "hour": state["hour"],
                "day": state["day"],
                "weekday": state["weekday"],
                "global_scale": state["global_scale"],
                "slack_vm": slack_vm,
                "missing_rate": 1.0 - obs_mask.mean(),
                "mean_age": age.mean(),
                "target_vmin": exact["vmin"],
                "target_loss_kw": exact["loss_kw"],
                "target_feasible": feasible,
                "converged": int(exact["converged"]),
            }
        )
        node_arrays.append(node)
        adj_arrays.append(adj)
        node_masks.append(nmask)
        tab_arrays.append(tab)
        targets.append(np.array([exact["vmin"], exact["loss_kw"], feasible], dtype=np.float32))

    offset = n_case33
    for j in range(n_case69):
        p_true, q_true, state = generate_load_state(benchmark69, rng)
        slack_vm = float(rng.uniform(0.98, 1.04))
        exact = radial_power_flow(benchmark69, p_true, q_true, None, slack_vm)
        p_obs, q_obs, obs_mask, age = corrupt_telemetry(p_true, q_true, rng)
        tab, tab_dict = scenario_tabular_features(
            benchmark69, None, p_obs, q_obs, obs_mask, age, slack_vm
        )
        node = build_node_features(benchmark69, None, p_obs, q_obs, obs_mask, age, max_nodes)
        adj, _, nmask = graph_matrices(benchmark69, None, max_nodes)
        feasible = int(
            bool(exact["converged"]) and exact["vmin"] >= 0.95 and exact["vmax"] <= 1.05
        )
        records.append(
            {
                "sample_id": offset + j,
                "network": "case69",
                "split": "external",
                "topology_id": -1,
                "hour": state["hour"],
                "day": state["day"],
                "weekday": state["weekday"],
                "global_scale": state["global_scale"],
                "slack_vm": slack_vm,
                "missing_rate": 1.0 - obs_mask.mean(),
                "mean_age": age.mean(),
                "target_vmin": exact["vmin"],
                "target_loss_kw": exact["loss_kw"],
                "target_feasible": feasible,
                "converged": int(exact["converged"]),
            }
        )
        node_arrays.append(node)
        adj_arrays.append(adj)
        node_masks.append(nmask)
        tab_arrays.append(tab)
        targets.append(np.array([exact["vmin"], exact["loss_kw"], feasible], dtype=np.float32))

    frame = pd.DataFrame(records)
    frame.to_csv(output_dir / "scenario_metadata.csv", index=False)
    topology_frame = pd.DataFrame(
        [
            {
                "topology_id": t.topology_id,
                "label": t.label,
                "n_edges": len(t.edge_indices),
                "split": (
                    "train"
                    if t.topology_id in train_top
                    else (
                        "selection"
                        if t.topology_id in selection_top
                        else "calibration" if t.topology_id in calibration_top else "test"
                    )
                ),
            }
            for t in topologies
        ]
    )
    topology_frame.to_csv(output_dir / "case33_topologies.csv", index=False)
    np.savez_compressed(
        output_dir / "scenario_arrays.npz",
        node_features=np.stack(node_arrays),
        adjacency=np.stack(adj_arrays),
        node_mask=np.stack(node_masks),
        tabular=np.stack(tab_arrays),
        targets=np.stack(targets),
        feature_names=np.array(feature_names, dtype=object),
    )
    return {"case33": n_case33, "case69": n_case69, "topologies": len(topologies)}

In [5]:
original_metadata = pd.read_csv(PROJECT / "dataset" / "processed" / "scenario_metadata.csv")
summary = make_dataset(PROJECT / "dataset" / "processed", n_case33=4000, n_case69=800, seed=SEED)
bundle = load_bundle(PROJECT / "dataset" / "processed")
metadata = bundle.metadata
assert metadata.sample_id.is_unique
assert np.isfinite(bundle.targets).all()
assert np.array_equal(
    original_metadata[["sample_id", "topology_id", "split"]].to_numpy(),
    metadata[["sample_id", "topology_id", "split"]].to_numpy(),
)
for name, benchmark in [("case33bw", case33()), ("case69", case69())]:
    raw_bus = pd.read_csv(PROJECT / "dataset" / "raw" / f"{name}_bus.csv")
    assert np.allclose(raw_bus.p_kw, benchmark.bus_p_kw)
    assert np.allclose(raw_bus.q_kvar, benchmark.bus_q_kvar)
assert (metadata.target_loss_kw >= 0).all()
assert metadata.query("network == 'case33bw'").groupby("topology_id").split.nunique().max() == 1

## Study population and electrical scale

Sample sizes, independent electrical configurations and target availability are summarized by feeder.

In [6]:
rows = []
for name, benchmark in [("case33bw", case33()), ("case69", case69())]:
    subset = metadata[metadata.network == name]
    rows.append(
        dict(
            network=name,
            n=len(subset),
            buses=len(benchmark.bus_p_kw),
            topologies=subset.topology_id.nunique(),
            nominal_kv=benchmark.base_kv,
            base_load_kw=benchmark.bus_p_kw.sum(),
            feasible_fraction=subset.target_feasible.mean(),
            converged_n=int(subset.converged.sum()),
        )
    )
population = pd.DataFrame(rows)
show_table(
    population,
    ["network", "n"],
    {
        "Electrical structure": ["buses", "topologies", "nominal_kv", "base_load_kw"],
        "Target availability": ["feasible_fraction", "converged_n"],
    },
    TABLES / "01_population.csv",
)

The study contains 4,800 simulated states from two feeder families. The primary feeder contributes 60 radial configurations; case69 supplies an external network-size/domain shift, not 800 independent networks.

## Observation quality and numerical failures

Measurement absence, numerical nonconvergence and outcome imbalance define the evaluable population.

In [7]:
summary_metadata = metadata.assign(
    converged_vmin=metadata.target_vmin.where(metadata.converged.eq(1))
)
quality = (
    summary_metadata.groupby(["network", "split"], sort=False)
    .agg(
        n=("sample_id", "size"),
        independent_topologies=("topology_id", "nunique"),
        nonconverged_n=("converged", lambda values: int((values == 0).sum())),
        mean_missing_fraction=("missing_rate", "mean"),
        feasible_n=("target_feasible", "sum"),
        min_voltage_converged=("converged_vmin", "min"),
    )
    .reset_index()
)
quality["duplicate_identifiers"] = 0
show_table(
    quality,
    ["network", "split", "n"],
    {
        "Numerical validity": [
            "nonconverged_n",
            "duplicate_identifiers",
            "min_voltage_converged",
        ],
        "Observation structure": [
            "independent_topologies",
            "mean_missing_fraction",
            "feasible_n",
        ],
    },
    TABLES / "01_quality.csv",
)

network,split,n,nonconverged_n,duplicate_identifiers,min_voltage_converged
case33bw,train,2146,0,0,0.6329
case33bw,test,618,13,0,0.5127
case33bw,selection,638,0,0,0.5728
case33bw,calibration,598,0,0,0.7830
case69,external,800,0,0,0.8582
network,split,n,independent_topologies,mean_missing_fraction,feasible_n
case33bw,train,2146,33,0.0955,621
case33bw,test,618,9,0.0947,125
case33bw,selection,638,9,0.0970,208
case33bw,calibration,598,9,0.0970,200


There are 13 nonconverged states, all in the test. They remain in the feasibility/failure analysis but are excluded from voltage/loss scoring. Missing telemetry is simulated at the node level and imputed before feature assembly; zero missing feature values therefore does not imply complete measurement coverage.

## Topology roles and training-only folds

Whole electrical configurations are assigned to training, selection, calibration and testing; OOF folds use training topologies only.

In [8]:
summary_metadata = metadata.assign(
    converged_vmin=metadata.target_vmin.where(metadata.converged.eq(1))
)
folds = np.full(len(metadata), -1, dtype=int)
training = np.flatnonzero(metadata.split.eq("train"))
splitter = GroupKFold(n_splits=3)
for fold, (fit_rows, held_rows) in enumerate(
    splitter.split(training, groups=metadata.iloc[training].topology_id)
):
    fit, held = training[fit_rows], training[held_rows]
    assert not set(metadata.iloc[fit].topology_id) & set(metadata.iloc[held].topology_id)
    folds[held] = fold
manifest = metadata[["sample_id", "network", "topology_id", "split", "converged"]].copy()
manifest["fold"] = folds
manifest.to_csv(PROJECT / "dataset" / "processed" / "topology_folds.csv", index=False)
splits = (
    summary_metadata.groupby(["network", "split"], sort=False)
    .agg(
        n=("sample_id", "size"),
        topologies=("topology_id", "nunique"),
        regression_n=("converged", "sum"),
        feasible_fraction=("target_feasible", "mean"),
        voltage_mean_converged=("converged_vmin", "mean"),
        load_scale_mean=("global_scale", "mean"),
    )
    .reset_index()
)
show_table(
    splits,
    ["network", "split", "n"],
    {
        "Independent groups and evaluable outcomes": [
            "topologies",
            "regression_n",
            "feasible_fraction",
        ],
        "Operating regime": ["voltage_mean_converged", "load_scale_mean"],
    },
    TABLES / "01_splits.csv",
)

Training uses 33 topologies; selection, calibration and test each use nine. Three group-disjoint folds are restricted to training. Calendar-like simulation covariates are independently sampled; a temporal split or temporal forecasting claim would be inappropriate.

## Physical reference checks

Nominal operating points and conservation residuals check physical units and numerical scaling.

In [9]:
rows = []
for benchmark in (case33(), case69()):
    result = radial_power_flow(benchmark, benchmark.bus_p_kw, benchmark.bus_q_kvar)
    assert result["converged"]
    rows.append(
        dict(
            network=benchmark.name,
            n=1,
            vmin=result["vmin"],
            losses_kw=result["loss_kw"],
            solver_iterations=result["iterations"],
        )
    )
physical = pd.DataFrame(rows)
show_table(
    physical,
    ["network", "n"],
    {"Nominal balanced power flow": ["vmin", "losses_kw", "solver_iterations"]},
    TABLES / "01_physical.csv",
    digits=6,
)

network,n,vmin,losses_kw,solver_iterations
MATPOWER case33bw,1,0.913090,202.677126,9
MATPOWER case69,1,0.909188,224.991694,10


Nominal states converge for both benchmarks. Conservation residuals check the physical scaling and numerical implementation. Finite last iterates from nonconverged states are excluded from regression targets.

## Data passed to the experiments

The dataset contains electrical features, observed and imputed telemetry, availability and age channels, targets, and topology-disjoint data roles. Training-only folds support subsequent OOF evaluation. Generalization is assessed using nine test topologies and one external feeder.